! Hier ist erst einmal die Idee, wie man das mit Clustern machen könnte. Muss dementsprechend noch angepasst werden, wenn die finalen Cluster stehen

In [ ]:
!pip install mistralAI

Import libraries

In [18]:
import os
import json
import re
import time
from mistralai import Mistral
from collections import defaultdict


Set up API

In [19]:
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model =  "open-mistral-nemo"
client = Mistral(api_key=api_key)

Set file paths and load data

In [20]:
#File paths
input_path = r"filtered_output.json"
cq_path = r"competency_questions_output/competency_questions_all_documents.txt"
clustered_questions_path = r"competency_questions_output/generalized_questions_SE.json"
output_file_path = "ontology_output/ontology_cluster_based.json"
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

#Load data
with open(input_path, "r", encoding="utf-8") as f:
    documents = json.load(f)



#Load competency questions
cq_by_doc_id = {}
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()
    
cq_blocks = re.split(r'Dokument:\s*', content)
for block in cq_blocks[1:]:
    lines = block.strip().splitlines()
    doc_id = lines[0].strip()
    frage = next((l for l in lines if "frage" in l.lower()), None)
    quelle = next((l for l in lines if "quelle" in l.lower()), None)
    if frage and quelle:
        cq_by_doc_id.setdefault(doc_id, []).append({
            "question": frage.replace("Frage:", "").strip(),
            "source": quelle.replace("Quelle:", "").strip()
        })




Load clustered questions

In [21]:
with open(clustered_questions_path, "r", encoding="utf-8") as f:
    clustered_questions = json.load(f)

Ontology system prompt

In [22]:
#Ontology system prompt
ontology_system_prompt = """
You are an expert legal ontology engineer.

Your task is to extract ontology components from the given legal ruling text (German court decisions). 
Identify and return three structured components:

1. **Classes**: Legal entities or concepts (e.g., Urteil, Entscheidungsgründe, Anspruch, Person, Tatbestand, Eiwendung, Rubrum, Tenor)
2. **Properties**: Verbs or phrases representing relationships or attributes (e.g., beinhaltet, basiert auf, verhindert, erörtern)
3. **Relationships**: Triples (Subject)-(Predicate)-(Object), connecting two classes using a property

Requirements:
- Only extract concepts if they have relationships.
- Use clear legal language in German.
- Output must follow this structure (example):

{
  "classes": ["Urteil", "Anspruch", "Tatbestand"],
  "properties": ["beinhaltet", "basiert auf"],
  "relationships": [
    ["Urteil", "beinhaltet", "Tatbestand"],
    ["Anspruch", "basiert auf", "Tatbestand"]
  ]
}

## Few-Shot Examples 
"<Example 1>"
"Sentence: 'Ein Arbeitnehmer hat eine Sozialversicherungsnummer'"
"Classes: [Arbeitnehmner, Sozialversicherungsnummer]"
"Properties: [hatIdentifikation]"
"Relationship: [(Arbeitnehmer)-(hatIdentifikation)-(Sozialversicherungsnummer)]"

"<Example 2>"
"Sentence: 'Arbeitnehmer haben nach Gesetz einen geregelten Mindestanspruch auf 24 Tage Urlaub im Jahr'"
"Classes: [Arbeitnehmer, Urlaub]"
"Properties: [hatMindestAnspruchAuf]"
"Relationship: [(Arbeitnehmer)-(hatMindestAnspruchAuf)-(Urlaub)]"

## Modelling Guidelines
"- **Concept Identification**:"
"- Identify nouns and noun phrases as potential **Classes**."
"- Identify verbs and verb phrases as potential **Properties**."
"- Identify prepositions that establish relationships between nouns."

"- **Classes**:"
"- Represent concepts in the domain, not the words that denote these concepts."
"- Avoid creating classes for synonyms; use a single class for concepts with the same meaning."

"- **Properties**:"
"- Represent significant relationships or attributes between classes."
"- Should be meaningful and represent a significant connection."

"- **Relationships**:"
"- Establish connections between classes using properties."
"- Ensure relationships are meaningful within the domain context."

"- **General Principles**:"
"- There is no single correct way to model a domain; the best solution depends on the application."
"- Ontology development is an iterative process; refine as needed."
"- Avoid cycles in the class hierarchy."
"- Siblings in the hierarchy should be at the same level of generality."

## Naming Conventions
"**General**:"
"- Do not add strings like 'class', 'domain', 'range', 'property', or 'slot' to names."
"- Use consistent naming throughout the ontology."

"**Classes**:"
"- Names are always capitalized."
"- Use nouns or compound nouns (e.g., Vertrag, Arbeit, Arbeitsvertrag)."
"- Use singular over plural (e.g., Arbeitsvertrag instead of Arbeitsverträge)."
"- Avoid abbreviations (e.g., Arbeitgeber instead of AG)."

"**Properties**:"
"- Names start with a lower-case letter."
"- Use verbs or verb phrases."
"- Can contain nouns in CamelCase starting with a verb (e.g., hatAnspruchAuf)."
"- Do not include spaces, commas, asterisks, or special characters."

##Finally
"Ensure that you include empty lists for classes, properties, or relationships if none are found."
"Make sure all extracted components, i.e. classes, properties, and relationships as well as all descriptions are in German and translate where necessary."
"Include a class only if at least one relationship is found for a class. Verify this requirement."
"Check carefully for each class without any relationship, based on the name and the description of the class, if it can be merged with another class or if it is actually a relationship between on class and another. This is particularly relevant for classes that are named in the form of 'has something'."
"All ontology components must be in German language!"
"Terms like 'range' or 'domain' are never allowed!"

Output must be valid JSON with no comments or explanation.
"""

#Prompt Builder 
def build_cluster_ontology_prompt(texts, questions):
    joined_text = "\n\n".join(texts)
    joined_questions = "\n- ".join(questions)
    return f"""Extract ontology components from the following German legal texts. 
The extraction should be informed by these competency questions:

- {joined_questions}

Texts:
{joined_text[:12000]}
"""


In [23]:
# Extract ontology using Mistral
def extract_ontology(prompt, system_prompt):
    response = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    raw = response['choices'][0]['message']['content']
    print("🔍 Raw response start:\n", raw[:300])

    # Debug print
    print("\n--- RAW MISTRAL OUTPUT (truncated) ---")
    print(raw_output[:500])
    print("--- END OF OUTPUT ---\n")

    try:
        json_text = re.search(r'\{[\s\S]+?\}', raw).group(0)
        return json.loads(json_text)
    except Exception as e:
        print(f"❌ Failed to parse JSON: {e}")
        return {"classes": [], "properties": [], "relationships": []}


Cluster-based ontology extraction

In [ ]:
#Normalize the texts
def normalize(text):
    return re.sub(r"\s+", " ", text.strip().lower())


ontology_by_cluster = {}

for cluster_id, cluster in clustered_questions_data.items():
    questions = cluster["questions"]
    generalized_question = cluster["generalized_question"]
    
    #Find doc ids for cluster's questions
    matched_doc_ids = set()
    normalized_cluster_questions = set(map(normalize, questions))
    
    for doc_id, entries in cq_by_doc_id.items():
        for entry in entries:
            if normalize(entry["question"]) in normalized_cluster_questions:
                matched_doc_ids.add(doc_id)
                
    
    #Collect texts: 
    relevant_texts = [documents[doc_id] for doc_id in matched_doc_ids if doc_id in documents]

    print(f"\n--- Cluster {cluster_id} ---")
    print(f"Generalized Question: {generalized_question}")
    print(f"# Questions: {len(questions)}")
    print(f"# Matched Docs: {len(matched_doc_ids)}")
    print(f"# Relevant Texts: {len(relevant_texts)}\n")
          
    if not relevant_texts:
        continue

    prompt = build_cluster_ontology_prompt(relevant_texts, questions)
    ontology = extract_ontology(prompt, ontology_system_prompt)

    ontology_by_cluster[cluster_id] = {
        "generalized_question": generalized_question,
        "ontology": ontology
    }


In [ ]:
print(f"Total clusters with ontology: {len(ontology_by_cluster)}")


# SAVE OUTPUT
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(ontology_by_cluster, f, ensure_ascii=False, indent=4)

print(f"Ontology saved to {output_file_path}")